# EgyptGPT Autoresearch

Sets up and runs an autonomous AI research loop on this repo using Claude Code.
Based on Karpathy's [autoresearch](https://github.com/karpathy/autoresearch) pattern.

**What this does:** Claude Code reads `program.md`, then autonomously edits the model/training code,
trains for 5 minutes, evaluates, keeps improvements, discards regressions, and repeats forever.

**Requirements:** Colab GPU runtime, Anthropic API key.

## 1. Setup

In [ ]:
# Clone repo with submodules, install dependencies
%cd /content
!rm -rf EgyptGPT  # clean slate in case of previous failed clone
!git clone --recurse-submodules https://github.com/JLansey/EgyptGPT.git
%cd /content/EgyptGPT
!bash setup.sh

In [ ]:
# Mount Google Drive for persistent storage (results + cached data)
from google.colab import drive
drive.mount('/content/drive')

import os
drive_dir = '/content/drive/MyDrive/EgyptGPT_autoresearch'
os.makedirs(drive_dir, exist_ok=True)
print(f'Drive folder: {drive_dir}')

In [ ]:
# Prepare data — use cached version from Drive if available, otherwise generate + cache
import os, shutil

%cd /content/EgyptGPT
data_dir = 'data/egypt_char'
drive_data = '/content/drive/MyDrive/EgyptGPT_autoresearch/data'
data_files = ['train.bin', 'val.bin', 'meta.pkl']

# Check if cached data exists on Drive
if all(os.path.exists(os.path.join(drive_data, f)) for f in data_files):
    print('Loading cached data from Google Drive...')
    for f in data_files:
        shutil.copy(os.path.join(drive_data, f), os.path.join(data_dir, f))
    print('Done.')
else:
    print('No cached data found. Running prepare.py (first time only)...')
    !python data/egypt_char/prepare.py
    # Cache to Drive for next time
    os.makedirs(drive_data, exist_ok=True)
    for f in data_files:
        src = os.path.join(data_dir, f)
        if os.path.exists(src):
            shutil.copy(src, os.path.join(drive_data, f))
    print(f'Cached data to {drive_data}')

# Verify
for f in data_files:
    path = os.path.join(data_dir, f)
    size = os.path.getsize(path) if os.path.exists(path) else 'MISSING'
    print(f'{path}: {size}')

## 2. Set your Anthropic API key

You can either:
- Use Colab Secrets (recommended): Add `ANTHROPIC_API_KEY` in the key icon on the left sidebar
- Or paste it directly in the cell below

In [ ]:
import os

# Option A: Load from Colab Secrets (recommended)
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('Loaded API key from Colab Secrets')
except Exception:
    pass

# Option B: Paste directly (uncomment and fill in)
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

assert os.environ.get('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY not set!'
print('API key is set.')

In [ ]:
# Install Node.js and Claude Code
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs
!npm install -g @anthropic-ai/claude-code
!claude --version

In [ ]:
# Install tmux for session persistence
!sudo apt-get install -y tmux

## 3. Configure git (for the experiment commits)

In [ ]:
%cd /content/EgyptGPT
!git config user.email "autoresearch@colab"
!git config user.name "autoresearch"

## 4. Launch autoresearch

This starts Claude Code in a tmux session. It will:
1. Read `program.md`
2. Create an `autoresearch/` branch
3. Run the baseline, then iterate autonomously

Each experiment takes ~5 minutes. Expect ~12/hour, ~100 overnight.

**To watch progress:** Open the Colab terminal and run `tmux attach -t autoresearch`

**To check results:** `!cat /content/EgyptGPT/results.tsv`

**To stop:** `!tmux kill-session -t autoresearch`

In [ ]:
import subprocess, os

# Kill any existing session
subprocess.run(['tmux', 'kill-session', '-t', 'autoresearch'], capture_output=True)

# Build the prompt
prompt = """Read program.md in this directory. This is an autoresearch setup.

1. Create branch autoresearch/run1 from master
2. Verify data/egypt_char/train.bin and val.bin exist
3. Initialize results.tsv with the header row
4. Begin the experiment loop as described in program.md

After each experiment, copy results.tsv to /content/drive/MyDrive/EgyptGPT_autoresearch/results.tsv so the log survives session crashes.

Start with the baseline run, then iterate autonomously. Never stop."""

# Start tmux session with Claude Code
cmd = f'''cd /content/EgyptGPT && ANTHROPIC_API_KEY={os.environ["ANTHROPIC_API_KEY"]} claude -p {repr(prompt)} --dangerously-skip-permissions'''
subprocess.run(['tmux', 'new-session', '-d', '-s', 'autoresearch', cmd])

print('Autoresearch is running in tmux session.')
print()
print('To watch:  open Colab terminal, run: tmux attach -t autoresearch')
print('To check:  !cat /content/EgyptGPT/results.tsv')
print('To stop:   !tmux kill-session -t autoresearch')
print()
print('Results backed up to: /content/drive/MyDrive/EgyptGPT_autoresearch/results.tsv')

## 5. Monitor progress

In [ ]:
# Check results so far
!cat /content/EgyptGPT/results.tsv 2>/dev/null || echo 'No results yet.'

In [ ]:
# Check git log for kept experiments
!cd /content/EgyptGPT && git log --oneline -20

In [ ]:
# Check if autoresearch is still running
!tmux has-session -t autoresearch 2>/dev/null && echo 'Running' || echo 'Stopped'